# 03 — Landmark Modelling Dataset Construction (SoccerMon 2020)

This notebook constructs the canonical modelling table used by the downstream landmark analyses.

## Scientific objective

SoccerMon contains minute-resolution monitoring data, but the injury target is available only at the athlete-session level and does not include an exact within-session injury-onset timestamp. The correct observational unit for supervised evaluation is therefore the **athlete-session**, not the individual minute.

This notebook preserves the minute-resolution predictors only as an intermediate representation. It creates cumulative and trailing dynamic features using information available **no later than the current session minute**. Downstream modelling notebooks then select one row per athlete-session at fixed elapsed-time landmarks (10, 20, 30, 40, 50, and 60 minutes).

The resulting landmark score should be interpreted only as discrimination of the same session-level injury-associated indicator as progressively more within-session information becomes available. It is **not** a minute-specific injury probability, injury-onset localization method, or prospective injury alert.

## Feature families

The canonical feature families are:

- **PRE:** 14 contextual variables;
- **CUM:** 30 cumulative within-session features;
- **DYN:** 33 dynamic within-session features.

The primary representation used downstream is **CUM+DYN (63 features)**.

Auxiliary missingness indicators are retained in the exported table for descriptive auditing only and are not part of the paper's 14-variable PRE feature family.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 200)

print("pandas:", pd.__version__)
print("numpy:", np.__version__)


## 1. Repository paths

This notebook reads the canonical processed outputs created by Notebook 01 and writes the modelling table to `results/modelling_data/`.


In [ ]:
def find_project_root(start: Path) -> Path:
    """Resolve the repository root when executed from the root or notebooks directory."""
    start = start.resolve()

    if start.name.lower() == "notebooks":
        return start.parent

    return start


PROJECT_ROOT = find_project_root(Path.cwd())

PROCESSED_DIR = PROJECT_ROOT / "results" / "processed_data"
OUTPUT_DIR = PROJECT_ROOT / "results" / "modelling_data"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MINUTE_FILE = PROCESSED_DIR / "minute_2020_data.csv"
MASTER_FILE = PROCESSED_DIR / "master_session_2020.csv"
MODEL_FILE = OUTPUT_DIR / "model_df_2020.csv"

print("Project root:", PROJECT_ROOT)
print("Minute-level input:", MINUTE_FILE)
print("Master-session input:", MASTER_FILE)
print("Modelling output:", MODEL_FILE)

for required_file in [MINUTE_FILE, MASTER_FILE]:
    if not required_file.exists():
        raise FileNotFoundError(
            "Required processed input was not found. "
            "Run Notebook 01 successfully before Notebook 03. "
            f"Missing: {required_file}"
        )


## 2. Load canonical processed tables

In [ ]:
minute_2020_data = pd.read_csv(
    MINUTE_FILE,
    low_memory=False,
)

master_session_2020 = pd.read_csv(
    MASTER_FILE,
    low_memory=False,
)

# Remove legacy index columns if older processed files are supplied.
minute_2020_data = minute_2020_data.drop(
    columns=["Unnamed: 0"],
    errors="ignore",
)
master_session_2020 = master_session_2020.drop(
    columns=["Unnamed: 0"],
    errors="ignore",
)

required_minute_columns = {
    "player_name",
    "session_id",
    "date",
    "minute",
    "minute_idx",
    "speed_mean",
    "speed_max",
    "heart_rate_mean",
    "heart_rate_max",
    "hacc_mean",
    "inst_acc_impulse_mean",
    "inst_acc_impulse_max",
    "accl_x_std",
    "accl_y_std",
    "accl_z_std",
    "gyro_x_std",
    "gyro_y_std",
    "gyro_z_std",
}

required_master_columns = {
    "player_name",
    "session_id",
    "ctl28",
    "ctl42",
    "daily_load",
    "weekly_load",
    "acwr",
    "monotony",
    "strain",
    "sleep_duration",
    "sleep_quality",
    "fatigue",
    "mood",
    "readiness",
    "soreness",
    "stress",
    "injury",
    "illness",
}

missing_minute = required_minute_columns - set(minute_2020_data.columns)
missing_master = required_master_columns - set(master_session_2020.columns)

assert not missing_minute, (
    "Minute-level input is missing required columns: "
    f"{sorted(missing_minute)}"
)
assert not missing_master, (
    "Master-session input is missing required columns: "
    f"{sorted(missing_master)}"
)

print("Minute-level rows:", len(minute_2020_data))
print("Master athlete-sessions:", len(master_session_2020))


## 3. Normalize identifiers and time fields

The timestamp normalization below is deterministic and does not create or alter the session-level injury target.


In [ ]:
if "team" not in minute_2020_data.columns:
    minute_2020_data["team"] = (
        minute_2020_data["player_name"]
        .astype(str)
        .str.split("-", n=1)
        .str[0]
    )

minute_2020_data["date"] = pd.to_datetime(
    minute_2020_data["date"],
    errors="raise",
)

minute_2020_data["minute"] = pd.to_datetime(
    minute_2020_data["minute"],
    errors="raise",
)

minute_2020_data["minute_idx"] = pd.to_numeric(
    minute_2020_data["minute_idx"],
    errors="raise",
).astype(int)

# Ensure the timestamp uses the reconstructed calendar date while preserving
# the recorded time of day.
minute_2020_data["minute"] = (
    minute_2020_data["date"].dt.normalize()
    + (
        minute_2020_data["minute"]
        - minute_2020_data["minute"].dt.normalize()
    )
)

minute_2020_data = (
    minute_2020_data
    .sort_values(
        ["player_name", "session_id", "minute_idx", "minute"]
    )
    .reset_index(drop=True)
)

duplicate_key = [
    "player_name",
    "session_id",
    "minute_idx",
]

assert not minute_2020_data.duplicated(
    duplicate_key
).any(), (
    "Duplicate athlete-session-minute rows detected."
)

assert not master_session_2020.duplicated(
    ["player_name", "session_id"]
).any(), (
    "Duplicate athlete-session rows detected in the master table."
)

print("Identifier and timestamp checks passed.")


## 4. Join session-level context and target to minute-level observations

The join is many-to-one from minute observations to athlete-session records. The session-level outcome is repeated only as an intermediate storage field; downstream supervised evaluation uses exactly one row per athlete-session at each landmark.


In [ ]:
SESSION_COLUMNS = [
    "player_name",
    "session_id",
    "ctl28",
    "ctl42",
    "daily_load",
    "weekly_load",
    "acwr",
    "monotony",
    "strain",
    "sleep_duration",
    "sleep_quality",
    "fatigue",
    "mood",
    "readiness",
    "soreness",
    "stress",
    "injury",
    "illness",
]

final_df = minute_2020_data.merge(
    master_session_2020[SESSION_COLUMNS],
    on=["player_name", "session_id"],
    how="left",
    validate="m:1",
)

assert len(final_df) == len(minute_2020_data)

unmatched_target_rows = int(final_df["injury"].isna().sum())
assert unmatched_target_rows == 0, (
    "Minute rows without a matched athlete-session injury target: "
    f"{unmatched_target_rows}"
)

final_df["injury"] = final_df["injury"].astype(int)
final_df["illness"] = final_df["illness"].fillna(0).astype(int)

print("Joined rows:", len(final_df))
print("All minute observations matched to an athlete-session target.")


## 5. Define contextual features and auxiliary missingness indicators

Missingness indicators are retained for auditability but are not included in the paper's canonical PRE feature family.


In [ ]:
PRE_FEATURES = [
    "ctl28",
    "ctl42",
    "daily_load",
    "weekly_load",
    "acwr",
    "monotony",
    "strain",
    "sleep_duration",
    "sleep_quality",
    "fatigue",
    "mood",
    "readiness",
    "soreness",
    "stress",
]

LOAD_FEATURES = PRE_FEATURES[:7]
WELLNESS_FEATURES = PRE_FEATURES[7:]

assert len(PRE_FEATURES) == 14
assert len(set(PRE_FEATURES)) == 14

MISSING_INDICATOR_FEATURES = [
    f"{column}_missing"
    for column in PRE_FEATURES
]

for column in PRE_FEATURES:
    final_df[f"{column}_missing"] = (
        final_df[column]
        .isna()
        .astype(np.int8)
    )

missingness_audit = (
    final_df[PRE_FEATURES]
    .isna()
    .mean()
    .sort_values(ascending=False)
    .rename("missing_fraction")
    .reset_index()
    .rename(columns={"index": "feature"})
)

display(missingness_audit)

print("PRE feature count:", len(PRE_FEATURES))
print(
    "Auxiliary missingness-indicator count:",
    len(MISSING_INDICATOR_FEATURES),
)


## 6. Construct the minute-resolution feature-engineering base

Only the signal variables required by the canonical cumulative and dynamic representations are retained before temporal feature construction.


In [ ]:
SIGNALS = [
    "speed_mean",
    "heart_rate_mean",
    "hacc_mean",
    "inst_acc_impulse_mean",
    "accl_x_std",
    "accl_y_std",
    "accl_z_std",
    "gyro_x_std",
    "gyro_y_std",
    "gyro_z_std",
]

CURRENT_MINUTE_SPREAD_INPUTS = [
    "heart_rate_max",
    "speed_max",
    "inst_acc_impulse_max",
]

BASE_COLUMNS = (
    [
        "session_id",
        "player_name",
        "minute_idx",
    ]
    + SIGNALS
    + CURRENT_MINUTE_SPREAD_INPUTS
    + PRE_FEATURES
    + MISSING_INDICATOR_FEATURES
    + ["injury"]
)

model_df = final_df[BASE_COLUMNS].copy()

model_df = (
    model_df
    .sort_values(
        ["player_name", "session_id", "minute_idx"]
    )
    .reset_index(drop=True)
)

assert len(SIGNALS) == 10
assert len(model_df) == len(final_df)

print("Temporal feature-engineering base:", model_df.shape)


## 7. Cumulative features

For each signal, expanding mean, maximum, and standard deviation are computed **within athlete-session only**, from session start through the current row. No later minute is included.


In [ ]:
session_groups = model_df.groupby(
    ["player_name", "session_id"],
    sort=False,
)

CUMULATIVE_FEATURES = []

for column in SIGNALS:
    mean_name = f"{column}_cum_mean"
    max_name = f"{column}_cum_max"
    std_name = f"{column}_cum_std"

    model_df[mean_name] = (
        session_groups[column]
        .expanding(min_periods=1)
        .mean()
        .reset_index(level=[0, 1], drop=True)
    )

    model_df[max_name] = session_groups[column].cummax()

    model_df[std_name] = (
        session_groups[column]
        .expanding(min_periods=2)
        .std()
        .reset_index(level=[0, 1], drop=True)
    )

    CUMULATIVE_FEATURES.extend(
        [mean_name, max_name, std_name]
    )

assert len(CUMULATIVE_FEATURES) == 30
assert len(set(CUMULATIVE_FEATURES)) == 30

print("Cumulative feature count:", len(CUMULATIVE_FEATURES))


## 8. Dynamic within-session features

Dynamic features consist of first differences, trailing five-observation rolling means and standard deviations, and three current-minute spread features.

All differences and rolling windows are grouped by athlete-session. Rolling windows are trailing rather than centered, so they cannot incorporate future rows.


In [ ]:
session_groups = model_df.groupby(
    ["player_name", "session_id"],
    sort=False,
)

DYNAMIC_FEATURES = []

for column in SIGNALS:
    delta_name = f"{column}_delta"
    roll_mean_name = f"{column}_roll_mean"
    roll_std_name = f"{column}_roll_std"

    model_df[delta_name] = session_groups[column].diff()

    model_df[roll_mean_name] = (
        session_groups[column]
        .rolling(window=5, min_periods=1)
        .mean()
        .reset_index(level=[0, 1], drop=True)
    )

    model_df[roll_std_name] = (
        session_groups[column]
        .rolling(window=5, min_periods=2)
        .std()
        .reset_index(level=[0, 1], drop=True)
    )

    DYNAMIC_FEATURES.extend(
        [
            delta_name,
            roll_mean_name,
            roll_std_name,
        ]
    )

model_df["hr_spike"] = (
    model_df["heart_rate_max"]
    - model_df["heart_rate_mean"]
)
model_df["speed_spike"] = (
    model_df["speed_max"]
    - model_df["speed_mean"]
)
model_df["impulse_spike"] = (
    model_df["inst_acc_impulse_max"]
    - model_df["inst_acc_impulse_mean"]
)

DYNAMIC_FEATURES.extend(
    [
        "hr_spike",
        "speed_spike",
        "impulse_spike",
    ]
)

assert len(DYNAMIC_FEATURES) == 33
assert len(set(DYNAMIC_FEATURES)) == 33

print("Dynamic feature count:", len(DYNAMIC_FEATURES))


## 9. Executable temporal-safety checks

The following checks recompute selected cumulative and rolling features directly from session prefixes and compare them with the stored feature values. This is an executable guard against accidental future leakage or cross-session contamination.


In [ ]:
def assert_close_or_nan(
    observed: float,
    expected: float,
    *,
    atol: float = 1e-12,
) -> None:
    """Assert numerical equality while treating paired NaNs as equal."""
    if pd.isna(observed) and pd.isna(expected):
        return

    assert np.isclose(
        observed,
        expected,
        atol=atol,
        rtol=0.0,
        equal_nan=True,
    ), (
        f"Temporal feature mismatch: observed={observed}, "
        f"expected={expected}"
    )


eligible_groups = [
    group
    for _, group in model_df.groupby(
        ["player_name", "session_id"],
        sort=False,
    )
    if len(group) >= 10
]

assert eligible_groups, "No sessions available for temporal-safety checks."

# Deterministic sample: first, middle, and last eligible session.
audit_groups = [
    eligible_groups[0],
    eligible_groups[len(eligible_groups) // 2],
    eligible_groups[-1],
]

for group in audit_groups:
    group = group.sort_values("minute_idx").reset_index(drop=True)

    positions = sorted(
        set(
            [
                0,
                min(4, len(group) - 1),
                min(9, len(group) - 1),
                len(group) - 1,
            ]
        )
    )

    for position in positions:
        prefix = group.iloc[: position + 1]

        for signal in SIGNALS:
            series = prefix[signal]

            assert_close_or_nan(
                group.loc[position, f"{signal}_cum_mean"],
                series.mean(),
            )
            assert_close_or_nan(
                group.loc[position, f"{signal}_cum_max"],
                series.max(),
            )
            assert_close_or_nan(
                group.loc[position, f"{signal}_cum_std"],
                series.std(ddof=1),
            )

            trailing = series.iloc[-5:]

            assert_close_or_nan(
                group.loc[position, f"{signal}_roll_mean"],
                trailing.mean(),
            )
            assert_close_or_nan(
                group.loc[position, f"{signal}_roll_std"],
                trailing.std(ddof=1),
            )

            expected_delta = (
                np.nan
                if position == 0
                else (
                    group.loc[position, signal]
                    - group.loc[position - 1, signal]
                )
            )

            assert_close_or_nan(
                group.loc[position, f"{signal}_delta"],
                expected_delta,
            )

print(
    "Temporal-safety audit passed: cumulative and dynamic "
    "features use only within-session history."
)


## 10. Assemble the canonical modelling table

The exported table retains identifiers, the 14 PRE variables, auxiliary missingness indicators, the 30 CUM variables, the 33 DYN variables, and the session-level injury-associated target.


In [ ]:
ID_COLUMNS = [
    "session_id",
    "minute_idx",
    "player_name",
]

PRIMARY_CUM_DYN_FEATURES = (
    CUMULATIVE_FEATURES
    + DYNAMIC_FEATURES
)

assert len(PRIMARY_CUM_DYN_FEATURES) == 63
assert len(set(PRIMARY_CUM_DYN_FEATURES)) == 63

MODEL_COLUMNS = (
    ID_COLUMNS
    + PRE_FEATURES
    + MISSING_INDICATOR_FEATURES
    + PRIMARY_CUM_DYN_FEATURES
    + ["injury"]
)

df_model = model_df[MODEL_COLUMNS].copy()

assert not df_model.duplicated(
    [
        "player_name",
        "session_id",
        "minute_idx",
    ]
).any()

print("Canonical modelling table shape:", df_model.shape)
print("PRE features:", len(PRE_FEATURES))
print("CUM features:", len(CUMULATIVE_FEATURES))
print("DYN features:", len(DYNAMIC_FEATURES))
print(
    "Primary CUM+DYN features:",
    len(PRIMARY_CUM_DYN_FEATURES),
)


## 11. Landmark availability and one-row-per-session contract

This notebook does not fit a model. It verifies that downstream landmark extraction can select at most one row per athlete-session at each fixed elapsed-time landmark.


In [ ]:
LANDMARKS = [10, 20, 30, 40, 50, 60]

landmark_rows = []

for landmark in LANDMARKS:
    landmark_df = df_model.loc[
        df_model["minute_idx"] == landmark
    ].copy()

    duplicate_sessions = int(
        landmark_df.duplicated(
            ["player_name", "session_id"]
        ).sum()
    )

    assert duplicate_sessions == 0, (
        f"Duplicate athlete-session observations at "
        f"{landmark} minutes: {duplicate_sessions}"
    )

    landmark_rows.append(
        {
            "landmark": landmark,
            "athlete_sessions": len(landmark_df),
            "athletes": landmark_df["player_name"].nunique(),
            "positive_sessions": int(
                landmark_df["injury"].sum()
            ),
            "positive_athletes": int(
                landmark_df.loc[
                    landmark_df["injury"] == 1,
                    "player_name",
                ].nunique()
            ),
        }
    )

landmark_availability = pd.DataFrame(landmark_rows)

display(landmark_availability)

assert (
    landmark_availability["positive_athletes"] == 5
).all(), (
    "A positive athlete is missing from at least one "
    "landmark-specific dataset."
)

print("Landmark unit-of-analysis checks passed.")


## 12. Frozen cohort and feature-identity audit

These assertions protect against silent changes in the reconstructed cohort or the primary 63-feature representation.


In [ ]:
session_outcomes = (
    df_model[
        [
            "player_name",
            "session_id",
            "injury",
        ]
    ]
    .drop_duplicates()
)

cohort_summary = {
    "minute_rows": int(len(df_model)),
    "athlete_sessions": int(len(session_outcomes)),
    "athletes": int(
        session_outcomes["player_name"].nunique()
    ),
    "positive_sessions": int(
        session_outcomes["injury"].sum()
    ),
    "positive_athletes": int(
        session_outcomes.loc[
            session_outcomes["injury"] == 1,
            "player_name",
        ].nunique()
    ),
}

EXPECTED_COHORT = {
    "minute_rows": 380_193,
    "athlete_sessions": 3_743,
    "athletes": 48,
    "positive_sessions": 22,
    "positive_athletes": 5,
}

assert cohort_summary == EXPECTED_COHORT, (
    "Frozen cohort accounting changed. "
    f"Observed: {cohort_summary}; "
    f"expected: {EXPECTED_COHORT}"
)

EXPECTED_PRIMARY_FEATURES = [
    "speed_mean_cum_mean",
    "speed_mean_cum_max",
    "speed_mean_cum_std",
    "heart_rate_mean_cum_mean",
    "heart_rate_mean_cum_max",
    "heart_rate_mean_cum_std",
    "hacc_mean_cum_mean",
    "hacc_mean_cum_max",
    "hacc_mean_cum_std",
    "inst_acc_impulse_mean_cum_mean",
    "inst_acc_impulse_mean_cum_max",
    "inst_acc_impulse_mean_cum_std",
    "accl_x_std_cum_mean",
    "accl_x_std_cum_max",
    "accl_x_std_cum_std",
    "accl_y_std_cum_mean",
    "accl_y_std_cum_max",
    "accl_y_std_cum_std",
    "accl_z_std_cum_mean",
    "accl_z_std_cum_max",
    "accl_z_std_cum_std",
    "gyro_x_std_cum_mean",
    "gyro_x_std_cum_max",
    "gyro_x_std_cum_std",
    "gyro_y_std_cum_mean",
    "gyro_y_std_cum_max",
    "gyro_y_std_cum_std",
    "gyro_z_std_cum_mean",
    "gyro_z_std_cum_max",
    "gyro_z_std_cum_std",
    "speed_mean_delta",
    "speed_mean_roll_mean",
    "speed_mean_roll_std",
    "heart_rate_mean_delta",
    "heart_rate_mean_roll_mean",
    "heart_rate_mean_roll_std",
    "hacc_mean_delta",
    "hacc_mean_roll_mean",
    "hacc_mean_roll_std",
    "inst_acc_impulse_mean_delta",
    "inst_acc_impulse_mean_roll_mean",
    "inst_acc_impulse_mean_roll_std",
    "accl_x_std_delta",
    "accl_x_std_roll_mean",
    "accl_x_std_roll_std",
    "accl_y_std_delta",
    "accl_y_std_roll_mean",
    "accl_y_std_roll_std",
    "accl_z_std_delta",
    "accl_z_std_roll_mean",
    "accl_z_std_roll_std",
    "gyro_x_std_delta",
    "gyro_x_std_roll_mean",
    "gyro_x_std_roll_std",
    "gyro_y_std_delta",
    "gyro_y_std_roll_mean",
    "gyro_y_std_roll_std",
    "gyro_z_std_delta",
    "gyro_z_std_roll_mean",
    "gyro_z_std_roll_std",
    "hr_spike",
    "speed_spike",
    "impulse_spike",
]

assert PRIMARY_CUM_DYN_FEATURES == EXPECTED_PRIMARY_FEATURES, (
    "Primary 63-feature identity or ordering changed."
)

print("=== FROZEN MODELLING RESOURCE ===")
for key, value in cohort_summary.items():
    print(f"{key}: {value:,}")

print("Primary feature identity/order: exact match (63 features)")


## 13. Export canonical modelling artifacts

In [ ]:
FEATURE_FILE = (
    OUTPUT_DIR
    / "primary_cum_dyn_features_2020.csv"
)

LANDMARK_FILE = (
    OUTPUT_DIR
    / "landmark_availability_2020.csv"
)

df_model.to_csv(
    MODEL_FILE,
    index=False,
)

pd.DataFrame(
    {
        "feature_order": range(
            1,
            len(PRIMARY_CUM_DYN_FEATURES) + 1,
        ),
        "feature_name": PRIMARY_CUM_DYN_FEATURES,
    }
).to_csv(
    FEATURE_FILE,
    index=False,
)

landmark_availability.to_csv(
    LANDMARK_FILE,
    index=False,
)

print("Saved:", MODEL_FILE)
print("Saved:", FEATURE_FILE)
print("Saved:", LANDMARK_FILE)


## Output contract

A successful run produces:

- `results/modelling_data/model_df_2020.csv`
- `results/modelling_data/primary_cum_dyn_features_2020.csv`
- `results/modelling_data/landmark_availability_2020.csv`

The modelling table contains minute-resolution intermediate rows, but downstream supervised analyses must select exactly one observation per athlete-session at a predefined landmark. The canonical primary representation contains 63 ordered CUM+DYN features constructed without observations from later minutes or other athlete-sessions.

No result produced by this notebook should be interpreted as localizing injury onset or estimating a minute-specific prospective injury probability.
